# 04 — Does deprivation explain the crime-rate spread?

Same test as London notebook 04, same caveats apply here too:

- IMD's composite score includes a Crime sub-domain (~9.3% weight) built
  partly from recorded police data, so we cross-check against the Income
  domain alone (no crime data in it).
- We use MHCLG's pre-aggregated local-authority-level IMD file rather than
  reconciling LSOA boundary vintages directly, since West Mercia's 9
  districts haven't changed boundaries either.

In [ ]:
import sys
sys.path.append("../../src")

import matplotlib.pyplot as plt
import seaborn as sns

from load_data import load_force_data, load_population, load_deprivation
from clean import clean_crime_data, add_area_column, WEST_MERCIA_DISTRICTS

sns.set_theme(style="whitegrid")

wm = load_force_data("west-mercia")
wm = clean_crime_data(wm)
wm = add_area_column(wm, column_name="District")
wmd = wm[wm["District"].isin(WEST_MERCIA_DISTRICTS)]

pop = load_population(WEST_MERCIA_DISTRICTS, name_column="District")
dep = load_deprivation(WEST_MERCIA_DISTRICTS, name_column="District")

counts = wmd.groupby("District").size().rename("Crimes").reset_index()
merged = (
    counts
    .merge(pop, on="District", validate="one_to_one")
    .merge(dep, on="District", validate="one_to_one")
)
merged["rate_per_1000"] = merged["Crimes"] / merged["Population"] * 1000
merged.sort_values("IMD_score", ascending=False)

## Chart — deprivation vs. per-capita crime rate

Only 9 districts and no extreme outlier this time (unlike London's
Westminster/City of London), so every point gets its own color-free label
— no need for the "highlight two outlier points" treatment from London's
notebook 04.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(merged["IMD_score"], merged["rate_per_1000"], color="#4C72B0", s=50)
for _, row in merged.iterrows():
    ax.annotate(
        row["District"], (row["IMD_score"], row["rate_per_1000"]),
        textcoords="offset points", xytext=(6, 4), fontsize=8,
    )
ax.set_xlabel("IMD 2019 average score (higher = more deprived)")
ax.set_ylabel("Recorded crimes per 1,000 residents (annual)")
ax.set_title("Deprivation vs. per-capita crime rate, West Mercia districts")
fig.tight_layout()

In [ ]:
print(f"IMD_score    corr: {merged['rate_per_1000'].corr(merged['IMD_score']):.3f}")
print(f"Income_score corr: {merged['rate_per_1000'].corr(merged['Income_score']):.3f}")

### Findings

- **Correlation is strong right away — 0.69 (IMD) and 0.71 (Income) —
  with no outliers excluded.** Compare to London, where the raw correlation
  across all 33 boroughs was only ~0.26, and only reached ~0.57 after
  dropping Westminster and City of London. West Mercia shows a cleaner
  deprivation-crime relationship from the start, most likely *because* it
  lacks a Westminster-scale non-resident-footfall outlier to distort it.
- The two most deprived districts (Telford and Wrekin, Redditch) both sit
  in the upper half of crime rates, and the three least deprived
  (Bromsgrove, Malvern Hills, Shropshire) sit at the bottom — a fairly
  orderly relationship.
- **Worcester is the one district that doesn't fit the trend** — only
  moderately deprived (4th out of 9) but the highest crime rate by some
  margin. Consistent with the footfall theory from notebook 03: it's the
  one district behaving like a small-scale Westminster.
- Same causality caveat as London: this is 9 data points at one point in
  time. Consistent with deprivation driving crime, but not proof of it —
  and with only 9 districts, a single unusual one (Worcester) has much more
  influence on the correlation than any single borough did in London's
  33-point version.